# Chapter 9 — Healthcare Data Quality Visualization (v2026)

> **LangChain 1.x / 2026 refresh.** Pinned versions, optional LangSmith tracing, and reproducible synthetic data. **Research-support only; synthetic/de-identified data; no clinical decisions.**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IvanReznikov/LangChain4LifeSciencesHealthcare/blob/main/notebooks/Chapter%2009.%20LangChain%20for%20Medicine%20and%20Healthcare/LC4LSH_Chapter_9_Healthcare_Data_Quality_Visualization.ipynb)

Profile a synthetic patient-visits dataset: cohort flow, temporal gaps, duplicates, impossible values, terminology ambiguity, and leakage checks. Uses the repo's synthetic CSVs and runs fully offline.

**Learning objectives**
- Load the synthetic patients/visits CSVs and build a cohort-flow summary.
- Detect duplicate records, impossible values, and temporal gaps.
- Flag terminology ambiguity and train/test leakage risks.
- Visualize distributions and gaps with matplotlib.

> **Runtime / cost / data.** Runs locally by default. Optional LLM cells are gated and can be skipped. **Synthetic / de-identified data only. Not for diagnosis, triage, treatment, dosage, prescribing, final billing/coding, EHR writes, or patient messaging.**


## Environment setup

Standard preamble so every chapter notebook starts the same way.


### Secrets

Keys are read from Colab Secrets if available, else from a local `.env`. All optional — the notebook runs without them (LLM cells are skipped).


In [ ]:
import os

try:
    from google.colab import userdata  # type: ignore

    def get_secret(name, default=""):
        return userdata.get(name) or default
except Exception:
    try:
        from dotenv import load_dotenv  # type: ignore

        load_dotenv()
    except Exception:
        pass

    def get_secret(name, default=""):
        return os.environ.get(name, default)


OPENAI_API_KEY = get_secret("LC4LS_OPENAI_API_KEY", "")
if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("OpenAI key set:", bool(OPENAI_API_KEY))


### Install pinned dependencies

Pinned versions keep the notebook reproducible. See `UPDATE_2026.md`.


In [ ]:
# Pinned versions - Last validated: 2026-07-21 (see UPDATE_2026.md)
%pip install -q "langchain==1.0.0" "langchain-core==1.2.30" "langchain-openai==1.0.0" "langchain-community==0.4" "pydantic>=2.5" "pandas" "matplotlib"


### Optional LangSmith tracing

Set `LANGCHAIN_API_KEY` to enable tracing of any LLM calls.


In [ ]:
import os

LANGSMITH_API_KEY = get_secret("LANGCHAIN_API_KEY", "")
LANGSMITH_PROJECT = "lc4lsh-chapter9-healthcare-data-quality"
if LANGSMITH_API_KEY.startswith(("lsv2_", "ls__")):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
    os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT
    print("LangSmith tracing ON ->", LANGSMITH_PROJECT)
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith tracing OFF")


## Load synthetic data

Uses the repo's synthetic datasets (no PHI). Paths fall back to a small in-memory frame if files are absent.

In [ ]:
import os
import pandas as pd

DATA_DIR = os.path.join("..", "..", "data", "datasets")
VISITS = os.path.join(DATA_DIR, "ch10_patients_visits_data.csv")

if os.path.exists(VISITS):
    visits = pd.read_csv(VISITS)
else:
    visits = pd.DataFrame({
        "Patient_ID": [1, 1, 2, 2, 2, 3],
        "Name": ["John Doe", "John Doe", "Jane Smith", "Jane Smith", "Jane Smith", "Bob Johnson"],
        "Age": [35, 35, 28, 28, 28, 42],
        "Diagnosis": ["Hypertension", "Hypertension", "Asthma", "Asthma", "Asthma", "Diabetes"],
        "Session": [1, 2, 1, 2, 3, 1],
        "Date": ["2022-01-01", "2022-01-15", "2022-02-01", "2022-02-15", "2022-03-01", "2022-03-15"],
    })

visits["Date"] = pd.to_datetime(visits["Date"], errors="coerce")
print(visits.shape)
visits.head()


### Cohort flow

How many unique patients, visits per patient, and sessions.

In [ ]:
n_patients = visits["Patient_ID"].nunique()
visits_per_patient = visits.groupby("Patient_ID").size()

print("unique patients:", n_patients)
print("total visits:", len(visits))
print("visits per patient:")
print(visits_per_patient.describe())


### Duplicate records

Exact-duplicate rows and same patient+date duplicates.

In [ ]:
exact_dupes = visits.duplicated().sum()
key_dupes = visits.duplicated(subset=["Patient_ID", "Date"]).sum()
print("exact duplicate rows:", int(exact_dupes))
print("duplicate patient+date rows:", int(key_dupes))


### Impossible values

Range checks on numeric fields (illustrative thresholds).

In [ ]:
issues = []
if "Age" in visits:
    bad_age = visits[(visits["Age"] < 0) | (visits["Age"] > 120)]
    issues.append(("age_out_of_range", len(bad_age)))
null_dates = int(visits["Date"].isna().sum())
issues.append(("unparseable_dates", null_dates))

for name, n in issues:
    print(f"{name}: {n}")


### Temporal gaps

Days between consecutive visits per patient.

In [ ]:
import matplotlib.pyplot as plt

v = visits.dropna(subset=["Date"]).sort_values(["Patient_ID", "Date"])
v["gap_days"] = v.groupby("Patient_ID")["Date"].diff().dt.days
gaps = v["gap_days"].dropna()

print("median gap (days):", gaps.median() if len(gaps) else "n/a")
if len(gaps):
    gaps.plot(kind="hist", bins=10, title="Inter-visit gap (days)")
    plt.xlabel("days")
    plt.show()


### Terminology ambiguity

Free-text diagnosis variants that likely map to one concept.

In [ ]:
if "Diagnosis" in visits:
    print(visits["Diagnosis"].value_counts())
# e.g. "High cholesterol" vs "high cholesterol" vs "Hypercholesterolemia" -> one concept


### Leakage check

If you split by visit (not by patient), the same patient can appear in train and test.

In [ ]:
patients = visits["Patient_ID"].unique().tolist()
split = int(len(patients) * 0.7)
train_p, test_p = set(patients[:split]), set(patients[split:])
overlap = train_p & test_p
print("patient leakage across a patient-wise split:", overlap if overlap else "none")

row_split = int(len(visits) * 0.7)
train_rows = set(visits.iloc[:row_split]["Patient_ID"])
test_rows = set(visits.iloc[row_split:]["Patient_ID"])
print("patient overlap with naive row split:", train_rows & test_rows)


### Quality report

A compact dict you can persist or feed to a dashboard.

In [ ]:
import json

report = {
    "n_patients": int(n_patients),
    "n_visits": int(len(visits)),
    "exact_duplicate_rows": int(exact_dupes),
    "duplicate_patient_date_rows": int(key_dupes),
    "unparseable_dates": int(null_dates),
}
with open("data_quality_report.json", "w") as f:
    json.dump(report, f, indent=2)
print(report)


## Limitations & safety

- **Research-support only.** Not for diagnosis, triage, treatment recommendation, dosage, prescribing, final billing/coding, EHR writes, or patient messaging.
- **Synthetic / de-identified data only.** Real PHI requires governance, BAA-covered infrastructure, and access controls.
- Optional LLM outputs are **drafts for human review** and can hallucinate — always verify against source data.
- Any scoring/eligibility logic here is illustrative and must be validated by qualified clinicians before any real use.


In [ ]:
# Cleanup: drop references and free memory.
import gc

for _name in ["llm", "chain", "model"]:
    globals().pop(_name, None)

gc.collect()
print("Cleanup complete.")


## Exercises

<details><summary>Q1. Why split train/test by patient rather than by row?</summary>
The same patient appears in multiple visits. A row-wise split leaks a patient's information across train and test, inflating performance. Splitting by patient keeps each patient in exactly one partition.
</details>

<details><summary>Q2. Why flag duplicate patient+date rows?</summary>
They usually indicate double-entry or an export artifact. Left in place they double-count a visit and bias utilization and gap statistics.
</details>

<details><summary>Q3. Why track terminology ambiguity separately from impossible values?</summary>
Impossible values are hard errors (age &lt; 0). Terminology ambiguity ("high cholesterol" vs "Hypercholesterolemia") is a normalization problem that requires mapping to a shared concept, not deletion.
</details>

### Task A — Concept normalization map
Build a small dictionary mapping free-text diagnosis variants to canonical concepts and show the cleaned value counts.

### Task B — Missingness matrix
Add a per-column missingness table and bar chart across all fields.

### Task C — Visit-interval outliers
Flag inter-visit gaps beyond a chosen percentile as potential data-entry or follow-up anomalies.

### Task D — Group-aware split helper
Write a function that returns train/test indices grouped by Patient_ID (a poor-man's GroupShuffleSplit) and verify zero patient overlap.
